執行sql/01_data_cleaning.sql前需先在 BigQuery 建立目標 dataset

In [ ]:
CREATE SCHEMA IF NOT EXISTS traffic_ad_roi_clean;

| 步驟                           | 目的                           |
| ---------------------------- | ---------------------------- |
| Step 1 — Validation Checks   | 執行前先檢查，輸出 QA 報表              |
| Step 2 — Clean & Standardise | 清洗後寫入 traffic_ad_roi_clean.* |
| Step 3 — Summary Report      | 對比 raw vs clean 行數，確認結果      |

清洗邏輯說明
每張表均涵蓋以下清洗處理：

去重：ROW_NUMBER() OVER (PARTITION BY <pk>) 保留最新一筆，處理重複 primary key

NULL / 空值處理：COALESCE + NULLIF(TRIM(...), '') 統一填補預設值

Channel 標準化：CASE UPPER(TRIM(channel)) 將 GOOGLE、GOOGLE ADS 等變體統一為受控詞彙，對應你的 campaigns 表原有值

負數數值修正：GREATEST(..., 0) 防止 impressions、clicks、spend_usd 出現負數

CTR 重算：從原始 clicks/impressions 重新計算，比直接信任儲存值更可靠

衍生欄位：新增 is_active（campaigns）、cost_per_click_usd（ad_impressions）、engagement_tier（sessions）、order_value_tier（conversions）方便下游分析

***

## 以下是完整分析。

## 資料規模總覽

| 表名 | 行數 | 類型 |
|---|---|---|
| `sessions` | **511,797** | 事實表（最大） |
| `conversions` | 18,288 | 事實表 |
| `ad_impressions` | 3,720 | 事實表 |
| `campaigns` | **12** | 維度表（只有12個活動） |
| `v_campaign_daily_ctr_cvr` | 2,988 | 每日時序 view |
| `v_monthly_channel_trend` | 60 | 月度 view |
| `v_monthly_roi_trend` | 36 | 月度 ROI view |
| `v_campaign_roi` | 12 | 活動彙總 view |
| `v_campaign_ctr_cvr_scatter` | 10 | 散點圖 view |
| `v_campaign_type_roi` | 11 | 類型彙總 view |
| `v_channel_performance` | 5 | 渠道彙總 view |
| `v_device_channel_conversion` | 15 | 裝置×渠道 view |
| `v_ctr_bucket_analysis` | 6 | CTR 分桶 view |



***

## 12 個 Campaign 分析

資料涵蓋 2024 年全年，共 5 個渠道、12 個活動 ：

| 渠道 | 活動數 | 預算範圍/日 |
|---|---|---|
| Google Ads | 4 | $300–$800 |
| Facebook Ads | 3 | $350–$700 |
| Email | 3 | $30–$80 |
| Organic | 1 | $0 |
| Direct | 1 | $0 |

***

## 關鍵業務洞察（來自 v_campaign_roi & v_channel_performance）

### ROAS 排名（由高至低）

| 活動 | 渠道 | ROAS | ROI % |
|---|---|---|---|
| Email_Abandoned_Cart | Email | **45.12** | 4,412% |
| Email_Newsletter_Monthly | Email | 27.86 | 2,686% |
| Email_Promo_Flash_Sale | Email | 26.95 | 2,595% |
| Google_Shopping_Q1 | Google Ads | 3.54 | 254% |
| Facebook_Retargeting | Facebook Ads | 2.63 | 163% |
| Google_Nonbrand_Search | Google Ads | 2.17 | 117% |
| Google_Brand_Search | Google Ads | 2.12 | 112% |
| Facebook_Awareness | Facebook Ads | 1.08 | 8% |
| Facebook_Conversion | Facebook Ads | 1.04 | 4% |
| **Google_Display_Remarketing** | Google Ads | **0.70** | **-30%** ⚠️ |

> ⚠️ `Google_Display_Remarketing` 是唯一**虧損活動**（ROI -30%），值得在 Power BI 重點標示。

### 渠道效率對比 

| 渠道 | ROAS | CVR | CPA |
|---|---|---|---|
| Email | **30.8** | 7.4% | $3.36 |
| Google Ads | 2.15 | 3.5% | $44.45 |
| Facebook Ads | 1.38 | 2.7% | $64.08 |
| Direct | N/A | 6.8% | $0 |
| Organic | N/A | 1.9% | $0 |

***

## Power BI 報告頁面建議

基於以上數據，建議設計 **5 個報告頁面**：

| 頁面 | 主視覺 | 使用的表 |
|---|---|---|
| **1. Executive Summary** | KPI Cards (Revenue, ROAS, CVR, CPA) + 渠道 Donut | `v_channel_performance` |
| **2. Campaign ROI** | Bar chart ROAS排名 + 虧損警示 + Table | `v_campaign_roi` |
| **3. Channel Trend** | 月度折線圖 (Revenue + Spend) | `v_monthly_roi_trend`, `v_monthly_channel_trend` |
| **4. CTR & CVR 分析** | 散點圖 CTR vs CVR + CTR 分桶 Bar | `v_campaign_ctr_cvr_scatter`, `v_ctr_bucket_analysis` |
| **5. Device & Country** | Matrix 裝置×渠道 + 地圖/Bar | `v_device_channel_conversion`, `sessions` |

***

## Power BI Import 策略

- **Import 模式**：`campaigns`（12行，維度）、`v_campaign_roi`、`v_channel_performance`、`v_monthly_roi_trend`、`v_monthly_channel_trend`、`v_campaign_ctr_cvr_scatter`、`v_ctr_bucket_analysis`、`v_device_channel_conversion`、`v_campaign_type_roi`
- **視情況 Import**：`ad_impressions`（3,720行）、`conversions`（18,288行）
- **謹慎考慮**：`sessions`（511,797行）— 建議在 BigQuery 先聚合再 import，避免 Power BI 檔案過大



`sessions` 有 511,797 行，直接 import 會令 `.pbix` 很大且慢。以下是針對你的報告需求，預先在 BigQuery 聚合的 SQL。

## 需要哪些聚合維度？

根據你的 5 個報告頁面，`sessions` 主要用於：
- 渠道 × 裝置 × 國家的流量分佈
- 每月趨勢
- Bounce rate 和 engagement tier

***

## 建議建立 3 個聚合 View

### View 1：每日 × 渠道 × 裝置（替代原始 sessions）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_daily_summary` AS

SELECT
  session_date,
  channel,
  device,
  country,
  engagement_tier,
  COUNT(*)                              AS total_sessions,
  COUNTIF(is_bounce = 1)               AS bounced_sessions,
  SUM(pages_viewed)                    AS total_pages_viewed,
  SUM(session_duration_sec)            AS total_duration_sec,
  COUNTIF(engagement_tier = 'High')    AS high_engagement_sessions,
  COUNTIF(engagement_tier = 'Medium')  AS medium_engagement_sessions,
  COUNTIF(engagement_tier = 'Low')     AS low_engagement_sessions
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY
  session_date, channel, device, country, engagement_tier;



> 511,797 行 → 預計壓縮至約 **3,000–8,000 行**（視 country 數量而定）

***

### View 2：每月 × 渠道摘要（用於 Page 3 趨勢）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_monthly_summary` AS

SELECT
  FORMAT_DATE('%Y-%m', session_date)   AS year_month,
  channel,
  device,
  COUNT(*)                             AS total_sessions,
  COUNTIF(is_bounce = 1)              AS bounced_sessions,
  ROUND(AVG(session_duration_sec), 1) AS avg_duration_sec,
  ROUND(AVG(pages_viewed), 2)         AS avg_pages_viewed,
  COUNTIF(engagement_tier = 'High')   AS high_engagement_sessions
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY year_month, channel, device;



> 511,797 行 → 約 **200–400 行**

***

### View 3：國家 × 渠道摘要（用於地圖/國家分析）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_country_summary` AS

SELECT
  country,
  channel,
  device,
  COUNT(*)                             AS total_sessions,
  COUNTIF(is_bounce = 1)              AS bounced_sessions,
  ROUND(AVG(session_duration_sec), 1) AS avg_duration_sec
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY country, channel, device;


> 511,797 行 → 約 **100–300 行**

***

## 行數預估對比

| 表/View | 原始行數 | Import 後估計行數 | 壓縮率 |
|---|---|---|---|
| `sessions`（原始） | 511,797 | 511,797 | — |
| `v_sessions_daily_summary` | → | ~5,000 | ~99% |
| `v_sessions_monthly_summary` | → | ~300 | ~99.9% |
| `v_sessions_country_summary` | → | ~200 | ~99.9% |

***

## 執行建議

1. 在 BigQuery Console 逐條跑上面 3 條 SQL 建立 View
2. 確認 row count：

In [ ]:
SELECT 'v_sessions_daily_summary',   COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_daily_summary`
UNION ALL
SELECT 'v_sessions_monthly_summary', COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_monthly_summary`
UNION ALL
SELECT 'v_sessions_country_summary', COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_country_summary`;


3. 在 Power BI **import 這 3 個 view** 取代原始 `sessions` 表
4. 原始 `sessions` 完全不需要 import 到 Power BI

建完後告訴我 row count，我幫你繼續設定 Power BI 的 data model 關聯。